## In this notebook we will sample a further 4% of data to be used as training in the same way as before, and combine them to get a total of 9%, with the same validation data. We exclude the original training data and validation data before sampling.

### Loading required libraries and setting file paths and proportions:

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

# --------------------------------------------------
# PATHS
# --------------------------------------------------

SPECTRA_DIR = Path("spectra_train")
PREVIOUS_DATA_DIR = Path("model_datasets/stage_1_5pct")
OUTPUT_DIR = Path("model_datasets/stage_2_9pct")

# Keep the same output location and filenames expected by Experiment 3.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

SEED = 42
ADDITIONAL_SAMPLE_FRACTION = 0.05
TRAIN_FRACTION = 0.80

PART_FILES = [f"part-{i:03d}.parquet" for i in range(21)]
EXPECTED_PART_IDS = {f"{i:03d}" for i in range(21)}

print("Configuration loaded.")
print(f"Source directory: {SPECTRA_DIR}")
print(f"Previous split directory: {PREVIOUS_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Source files expected: {len(PART_FILES)}")


Configuration loaded.
Source directory: spectra_train
Previous split directory: model_datasets\stage_1_5pct
Output directory: model_datasets\stage_2_9pct
Source files expected: 21


### Creating canonical identifiers and row key helpers:

In [2]:
def normalise_source_part(value):
    """Convert 000, part-000, or part-000.parquet to canonical '000'."""
    if pd.isna(value):
        raise ValueError("Found a missing source_part value.")

    value = str(value).strip()
    value = Path(value).name

    if value.lower().endswith(".parquet"):
        value = value[:-8]

    if value.startswith("part-"):
        value = value[5:]

    # Source part IDs are expected to be three digits: 000 ... 020.
    if not re.fullmatch(r"\d{3}", value):
        raise ValueError(f"Unrecognised source_part value: {value!r}")

    return value


def standardise_source_part(df, dataset_name):
    if "source_part" not in df.columns:
        raise ValueError(f"{dataset_name} is missing source_part.")

    result = df.copy()
    result["source_part"] = result["source_part"].map(normalise_source_part)

    unexpected = set(result["source_part"].unique()) - EXPECTED_PART_IDS
    if unexpected:
        raise ValueError(
            f"{dataset_name} contains unexpected source parts: "
            f"{sorted(unexpected)[:10]}"
        )
    return result


def validate_row_identifiers(df, dataset_name):
    required = {"source_part", "row_id", "inchikey14"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{dataset_name} is missing columns: {sorted(missing)}")

    if df["source_part"].isna().any():
        raise ValueError(f"{dataset_name} contains missing source_part values.")
    if df["row_id"].isna().any():
        raise ValueError(f"{dataset_name} contains missing row_id values.")
    if df["inchikey14"].isna().any():
        raise ValueError(f"{dataset_name} contains missing inchikey14 labels.")

    # Use string keys consistently across all dataframes.
    return df


def make_row_keys(df):
    return set(zip(
        df["source_part"].map(normalise_source_part),
        df["row_id"].astype(str)
    ))


def duplicate_row_key_count(df):
    key_frame = pd.DataFrame({
        "source_part": df["source_part"].map(normalise_source_part),
        "row_id": df["row_id"].astype(str)
    })
    return int(key_frame.duplicated().sum())


print("Identifier helpers defined.")


Identifier helpers defined.


### Loading the data used in the previous experiments:

In [3]:
required_previous_files = [
    PREVIOUS_DATA_DIR / "train.parquet",
    PREVIOUS_DATA_DIR / "validation.parquet",
    PREVIOUS_DATA_DIR / "sampled_5pct.parquet",
    PREVIOUS_DATA_DIR / "class_mapping.json",
]

for path in required_previous_files:
    if not path.exists():
        raise FileNotFoundError(f"Required original file not found: {path}")

original_train = pd.read_parquet(PREVIOUS_DATA_DIR / "train.parquet")
original_validation = pd.read_parquet(PREVIOUS_DATA_DIR / "validation.parquet")
original_sampled = pd.read_parquet(PREVIOUS_DATA_DIR / "sampled_5pct.parquet")

# Standardise identifiers immediately, before any key comparisons.
original_train = standardise_source_part(original_train, "Original training")
original_validation = standardise_source_part(original_validation, "Original validation")
original_sampled = standardise_source_part(original_sampled, "Original sampled 5%")

for name, df in [
    ("Original training", original_train),
    ("Original validation", original_validation),
    ("Original sampled 5%", original_sampled),
]:
    validate_row_identifiers(df, name)
    duplicates = duplicate_row_key_count(df)
    if duplicates:
        raise AssertionError(f"{name} contains {duplicates:,} duplicate (source_part, row_id) keys.")

print(f"Original training spectra: {len(original_train):,}")
print(f"Original validation spectra: {len(original_validation):,}")
print(f"Original sampled spectra: {len(original_sampled):,}")
print("All original row identifiers are present and unique within each dataset.")


Original training spectra: 91,851
Original validation spectra: 22,963
Original sampled spectra: 114,814
All original row identifiers are present and unique within each dataset.


### Establishing and verifying the complete exclusion set:

In [4]:
original_train_keys = make_row_keys(original_train)
original_validation_keys = make_row_keys(original_validation)
original_sampled_keys = make_row_keys(original_sampled)

overlap_original_train_validation = original_train_keys & original_validation_keys
if overlap_original_train_validation:
    raise AssertionError(
        "The original training and validation datasets overlap: "
        f"{len(overlap_original_train_validation):,} rows."
    )

# The exclusion set is explicitly the union of original train AND validation.
previously_used_keys = original_train_keys | original_validation_keys

# Cross-check that sampled_5pct is exactly the union of those two splits.
missing_from_sampled_manifest = previously_used_keys - original_sampled_keys
extra_in_sampled_manifest = original_sampled_keys - previously_used_keys

print("Original split consistency checks")
print("----------------------------------")
print(f"Original train/validation overlap: {len(overlap_original_train_validation):,}")
print(f"Rows in original train + validation: {len(previously_used_keys):,}")
print(f"Rows in original sampled manifest: {len(original_sampled_keys):,}")
print(f"Split rows missing from sampled manifest: {len(missing_from_sampled_manifest):,}")
print(f"Extra rows in sampled manifest: {len(extra_in_sampled_manifest):,}")

if missing_from_sampled_manifest or extra_in_sampled_manifest:
    raise AssertionError(
        "The original train/validation split does not match sampled_5pct.parquet "
        "after source-part normalisation. Stop and investigate the original files."
    )

print("\nVerified: original sampled manifest equals original train + validation.")
print(f"Exclusion set contains {len(previously_used_keys):,} unique spectra.")


Original split consistency checks
----------------------------------
Original train/validation overlap: 0
Rows in original train + validation: 114,814
Rows in original sampled manifest: 114,814
Split rows missing from sampled manifest: 0
Extra rows in sampled manifest: 0

Verified: original sampled manifest equals original train + validation.
Exclusion set contains 114,814 unique spectra.


### Sampling the additional spectra, excluding every previously used row:

In [5]:
additional_sample_parts = []
additional_train_parts = []
additional_unused_parts = []
per_part_records = []

for part_number, filename in enumerate(PART_FILES):
    print(f"Processing {filename} ...")

    file_path = SPECTRA_DIR / filename
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    part_df = pd.read_parquet(file_path)

    if "row_id" not in part_df.columns or "inchikey14" not in part_df.columns:
        raise ValueError(f"{filename} must contain row_id and inchikey14.")

    # Canonical source ID is identical to the format used in all previous data.
    canonical_part_id = normalise_source_part(filename)
    part_df["source_part"] = canonical_part_id

    validate_row_identifiers(part_df, filename)

    source_keys = make_row_keys(part_df)
    if len(source_keys) != len(part_df):
        raise AssertionError(f"{filename} contains duplicate row keys.")

    # Exclude original training AND original validation rows.
    already_used_mask = np.fromiter(
        (
            (canonical_part_id, str(row_id)) in previously_used_keys
            for row_id in part_df["row_id"]
        ),
        dtype=bool,
        count=len(part_df)
    )

    remaining_df = part_df.loc[~already_used_mask].copy()

    # Independent, deterministic 5% sample from the remaining rows in this part.
    n_additional = round(len(remaining_df) * ADDITIONAL_SAMPLE_FRACTION)
    additional_sample = remaining_df.sample(
        n=n_additional,
        random_state=SEED + part_number
    ).copy()

    # Shuffle deterministically before taking the 80% training portion.
    additional_sample = additional_sample.sample(
        frac=1,
        random_state=SEED + 1000 + part_number
    ).reset_index(drop=True)

    n_train = round(len(additional_sample) * TRAIN_FRACTION)
    additional_train = additional_sample.iloc[:n_train].copy()
    additional_unused = additional_sample.iloc[n_train:].copy()

    additional_sample_parts.append(additional_sample)
    additional_train_parts.append(additional_train)
    additional_unused_parts.append(additional_unused)

    per_part_records.append({
        "source_part": canonical_part_id,
        "source_rows": len(part_df),
        "previously_used_rows_excluded": int(already_used_mask.sum()),
        "remaining_eligible_rows": len(remaining_df),
        "additional_sample_rows": len(additional_sample),
        "additional_train_rows": len(additional_train),
        "additional_unused_rows": len(additional_unused),
    })

    print(
        f"  source={len(part_df):,} | excluded={int(already_used_mask.sum()):,} "
        f"| eligible={len(remaining_df):,} | sampled={len(additional_sample):,} "
        f"| train={len(additional_train):,} | unused={len(additional_unused):,}"
    )

additional_sampled_df = pd.concat(additional_sample_parts, ignore_index=True)
additional_train_df = pd.concat(additional_train_parts, ignore_index=True)
additional_unused_df = pd.concat(additional_unused_parts, ignore_index=True)
per_part_summary = pd.DataFrame(per_part_records)

print("\nAdditional sampling complete.")
print(f"Additional sample: {len(additional_sampled_df):,}")
print(f"Additional training: {len(additional_train_df):,}")
print(f"Additional unused remainder: {len(additional_unused_df):,}")


Processing part-000.parquet ...
  source=126,233 | excluded=6,312 | eligible=119,921 | sampled=5,996 | train=4,797 | unused=1,199
Processing part-001.parquet ...
  source=126,164 | excluded=6,308 | eligible=119,856 | sampled=5,993 | train=4,794 | unused=1,199
Processing part-002.parquet ...
  source=126,305 | excluded=6,315 | eligible=119,990 | sampled=6,000 | train=4,800 | unused=1,200
Processing part-003.parquet ...
  source=126,606 | excluded=6,330 | eligible=120,276 | sampled=6,014 | train=4,811 | unused=1,203
Processing part-004.parquet ...
  source=126,387 | excluded=6,319 | eligible=120,068 | sampled=6,003 | train=4,802 | unused=1,201
Processing part-005.parquet ...
  source=126,432 | excluded=6,322 | eligible=120,110 | sampled=6,006 | train=4,805 | unused=1,201
Processing part-006.parquet ...
  source=126,337 | excluded=6,317 | eligible=120,020 | sampled=6,001 | train=4,801 | unused=1,200
Processing part-007.parquet ...
  source=126,579 | excluded=6,329 | eligible=120,250 | sam

### Performing integrity checks before combining the datasets:

In [6]:
# Validate row identifiers and uniqueness in all newly sampled datasets.
for name, df in [
    ("Additional sample", additional_sampled_df),
    ("Additional training", additional_train_df),
    ("Additional unused", additional_unused_df),
]:
    validate_row_identifiers(df, name)
    duplicates = duplicate_row_key_count(df)
    if duplicates:
        raise AssertionError(f"{name} contains {duplicates:,} duplicate row keys.")

additional_sample_keys = make_row_keys(additional_sampled_df)
additional_train_keys = make_row_keys(additional_train_df)
additional_unused_keys = make_row_keys(additional_unused_df)

# No additional sampled row may occur in original train OR original validation.
overlap_additional_original_train = additional_sample_keys & original_train_keys
overlap_additional_original_validation = additional_sample_keys & original_validation_keys
overlap_additional_original_sampled = additional_sample_keys & original_sampled_keys

# Train and unused portions must be disjoint and together equal the additional sample.
overlap_additional_train_unused = additional_train_keys & additional_unused_keys
additional_partition_union = additional_train_keys | additional_unused_keys

missing_from_additional_partition = additional_sample_keys - additional_partition_union
unexpected_in_additional_partition = additional_partition_union - additional_sample_keys

print("PRE-COMBINATION INTEGRITY CHECKS")
print("--------------------------------")
print(f"Additional sample vs original training: {len(overlap_additional_original_train):,}")
print(f"Additional sample vs original validation: {len(overlap_additional_original_validation):,}")
print(f"Additional sample vs original sampled manifest: {len(overlap_additional_original_sampled):,}")
print(f"Additional train/unused overlap: {len(overlap_additional_train_unused):,}")
print(f"Additional sample rows missing from train+unused: {len(missing_from_additional_partition):,}")
print(f"Unexpected rows in train+unused: {len(unexpected_in_additional_partition):,}")

assert len(overlap_additional_original_train) == 0
assert len(overlap_additional_original_validation) == 0
assert len(overlap_additional_original_sampled) == 0
assert len(overlap_additional_train_unused) == 0
assert len(missing_from_additional_partition) == 0
assert len(unexpected_in_additional_partition) == 0
assert len(additional_sampled_df) == len(additional_train_df) + len(additional_unused_df)

print("\nAll pre-combination integrity checks passed.")


PRE-COMBINATION INTEGRITY CHECKS
--------------------------------
Additional sample vs original training: 0
Additional sample vs original validation: 0
Additional sample vs original sampled manifest: 0
Additional train/unused overlap: 0
Additional sample rows missing from train+unused: 0
Unexpected rows in train+unused: 0

All pre-combination integrity checks passed.


### Creating the combined training dataset while preserving the validation data:

In [7]:
combined_train_df = pd.concat(
    [original_train, additional_train_df],
    ignore_index=True
)

# Keep the original validation rows and labels unchanged.
combined_validation_df = original_validation.copy()

combined_train_keys = make_row_keys(combined_train_df)
combined_validation_keys = make_row_keys(combined_validation_df)

overlap_combined_train_validation = combined_train_keys & combined_validation_keys
duplicate_combined_train = duplicate_row_key_count(combined_train_df)
duplicate_combined_validation = duplicate_row_key_count(combined_validation_df)

print("COMBINED DATASET INTEGRITY")
print("-------------------------")
print(f"Combined training spectra: {len(combined_train_df):,}")
print(f"Validation spectra: {len(combined_validation_df):,}")
print(f"Training/validation overlap: {len(overlap_combined_train_validation):,}")
print(f"Duplicate combined training rows: {duplicate_combined_train:,}")
print(f"Duplicate validation rows: {duplicate_combined_validation:,}")

assert len(overlap_combined_train_validation) == 0
assert duplicate_combined_train == 0
assert duplicate_combined_validation == 0
assert len(combined_validation_df) == len(original_validation)

print("\nCombined training and original validation are disjoint.")


COMBINED DATASET INTEGRITY
-------------------------
Combined training spectra: 179,108
Validation spectra: 22,963
Training/validation overlap: 0
Duplicate combined training rows: 0
Duplicate validation rows: 0

Combined training and original validation are disjoint.


### Extending the class mapping without changing existing indices:

In [8]:
with open(PREVIOUS_DATA_DIR / "class_mapping.json", "r") as f:
    original_mapping_json = json.load(f)


def extract_class_mapping(data):
    if not isinstance(data, dict):
        raise ValueError("Expected the original class mapping JSON to contain a dictionary.")

    for key in ("class_to_idx", "class_mapping"):
        if key in data and isinstance(data[key], dict):
            data = data[key]
            break
    else:
        if "idx_to_class" in data and isinstance(data["idx_to_class"], dict):
            return {
                str(label): int(index)
                for index, label in data["idx_to_class"].items()
            }

    if not all(isinstance(v, (int, np.integer)) for v in data.values()):
        raise ValueError("Could not identify a valid class-to-index mapping.")

    return {str(label): int(index) for label, index in data.items()}


original_class_to_idx = extract_class_mapping(original_mapping_json)

if not original_class_to_idx:
    raise ValueError("Original class mapping is empty.")

original_indices = list(original_class_to_idx.values())
if len(set(original_indices)) != len(original_indices):
    raise AssertionError("Original class mapping contains duplicate indices.")

if set(original_indices) != set(range(len(original_indices))):
    raise AssertionError("Original class indices are not contiguous from zero.")

training_classes = set(combined_train_df["inchikey14"].astype(str))
original_mapping_classes = set(original_class_to_idx)

# Preserve all original label indices and append any newly encountered labels.
updated_class_to_idx = original_class_to_idx.copy()
new_classes = sorted(training_classes - original_mapping_classes)

next_index = len(updated_class_to_idx)
for label in new_classes:
    updated_class_to_idx[label] = next_index
    next_index += 1

# Final mapping integrity checks.
updated_indices = list(updated_class_to_idx.values())
assert len(updated_indices) == len(set(updated_indices))
assert set(updated_indices) == set(range(len(updated_indices)))
assert training_classes.issubset(set(updated_class_to_idx))

# Explicitly verify that no old index changed.
for label, old_index in original_class_to_idx.items():
    assert updated_class_to_idx[label] == old_index

validation_classes = set(combined_validation_df["inchikey14"].astype(str))
unmapped_validation = validation_classes - set(updated_class_to_idx)

print(f"Original classes: {len(original_class_to_idx):,}")
print(f"New classes added: {len(new_classes):,}")
print(f"Updated training classes: {len(updated_class_to_idx):,}")
print(f"Validation classes not represented in training: {len(unmapped_validation):,}")
print("Original class indices preserved; updated mapping is unique and contiguous.")


Original classes: 65,644
New classes added: 42,218
Updated training classes: 107,862
Validation classes not represented in training: 9,081
Original class indices preserved; updated mapping is unique and contiguous.


### Building a complete split manifest:

In [9]:
original_manifest = original_sampled[
    ["source_part", "row_id", "inchikey14"]
].copy()

original_train_key_set = make_row_keys(original_train)
original_manifest_keys = list(zip(
    original_manifest["source_part"].map(normalise_source_part),
    original_manifest["row_id"].astype(str)
))
original_manifest["split"] = [
    "train" if key in original_train_key_set else "validation"
    for key in original_manifest_keys
]

additional_train_manifest = additional_train_df[
    ["source_part", "row_id", "inchikey14"]
].copy()
additional_train_manifest["split"] = "additional_train"

additional_unused_manifest = additional_unused_df[
    ["source_part", "row_id", "inchikey14"]
].copy()
additional_unused_manifest["split"] = "unused"

split_manifest = pd.concat(
    [original_manifest, additional_train_manifest, additional_unused_manifest],
    ignore_index=True
)

if duplicate_row_key_count(split_manifest):
    raise AssertionError("Split manifest contains duplicate row keys.")

manifest_keys = make_row_keys(split_manifest)
expected_manifest_keys = original_sampled_keys | additional_sample_keys

if manifest_keys != expected_manifest_keys:
    raise AssertionError("Split manifest does not cover exactly the original and additional sampled rows.")

print("Manifest split counts:")
print(split_manifest["split"].value_counts())
print(f"Total manifest rows: {len(split_manifest):,}")


Manifest split counts:
split
train               91851
additional_train    87257
validation          22963
unused              21815
Name: count, dtype: int64
Total manifest rows: 223,886


### Saving datasets:

In [10]:
# Save combined training and unchanged original validation.
combined_train_df.to_parquet(OUTPUT_DIR / "train.parquet", index=False)
combined_validation_df.to_parquet(OUTPUT_DIR / "validation.parquet", index=False)

# Save additional sample components for auditability.
additional_sampled_df.to_parquet(OUTPUT_DIR / "additional_sampled.parquet", index=False)
additional_train_df.to_parquet(OUTPUT_DIR / "additional_train.parquet", index=False)
additional_unused_df.to_parquet(OUTPUT_DIR / "additional_unused.parquet", index=False)

split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
per_part_summary.to_csv(OUTPUT_DIR / "per_part_sampling_summary.csv", index=False)

with open(OUTPUT_DIR / "class_mapping.json", "w") as f:
    json.dump(updated_class_to_idx, f, indent=2)

experiment_config = {
    "experiment": "Fine-tuned DreaMS classification - Experiment 3",
    "seed": SEED,
    "original_sample_fraction_per_part": 0.05,
    "additional_sample_fraction_of_remaining_per_part": ADDITIONAL_SAMPLE_FRACTION,
    "additional_train_fraction": TRAIN_FRACTION,
    "original_training_spectra": len(original_train),
    "original_validation_spectra": len(original_validation),
    "additional_sampled_spectra": len(additional_sampled_df),
    "additional_training_spectra": len(additional_train_df),
    "additional_unused_spectra": len(additional_unused_df),
    "combined_training_spectra": len(combined_train_df),
    "validation_spectra": len(combined_validation_df),
    "original_training_classes": len(original_class_to_idx),
    "updated_training_classes": len(updated_class_to_idx),
    "validation_source": "Original Experiment 1 validation set",
    "validation_filtering": "Applied during model training using known training classes",
    "heldout_data_used": False,
    "source_part_key_format": "Canonical three-digit source IDs, e.g. 000",
    "exclusion_rule": "Union of original training and original validation (cross-checked against sampled_5pct.parquet)",
    "integrity_checks": "Passed in-memory and after-save row-key overlap checks"
}

with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

print("Datasets and metadata saved to:", OUTPUT_DIR.resolve())


Datasets and metadata saved to: C:\Users\micha\Downloads\enveda-CASMI26-molecule-id-mass-spectra-20261113T021650Z-1-003\enveda-CASMI26-molecule-id-mass-spectra\model_datasets\stage_2_9pct


### Re-loading saved files to independently verify no leakage:

In [11]:
# Reload the files that the Experiment 3 model notebook will actually consume.
saved_train = pd.read_parquet(OUTPUT_DIR / "train.parquet")
saved_validation = pd.read_parquet(OUTPUT_DIR / "validation.parquet")
saved_additional_sample = pd.read_parquet(OUTPUT_DIR / "additional_sampled.parquet")
saved_additional_train = pd.read_parquet(OUTPUT_DIR / "additional_train.parquet")
saved_additional_unused = pd.read_parquet(OUTPUT_DIR / "additional_unused.parquet")

for name, df in [
    ("Saved training", saved_train),
    ("Saved validation", saved_validation),
    ("Saved additional sample", saved_additional_sample),
    ("Saved additional training", saved_additional_train),
    ("Saved additional unused", saved_additional_unused),
]:
    validate_row_identifiers(df, name)

saved_train_keys = make_row_keys(saved_train)
saved_validation_keys = make_row_keys(saved_validation)
saved_additional_sample_keys = make_row_keys(saved_additional_sample)

saved_overlap_train_validation = saved_train_keys & saved_validation_keys
saved_overlap_additional_original = saved_additional_sample_keys & previously_used_keys
saved_duplicate_train = duplicate_row_key_count(saved_train)
saved_duplicate_validation = duplicate_row_key_count(saved_validation)

# Verify the saved validation set is exactly the original validation set by row key
# and label, not merely the same length.
def key_to_label(df):
    return {
        (normalise_source_part(part), str(row_id)): str(label)
        for part, row_id, label in zip(
            df["source_part"], df["row_id"], df["inchikey14"]
        )
    }

original_validation_labels = key_to_label(original_validation)
saved_validation_labels = key_to_label(saved_validation)

print("POST-SAVE INTEGRITY CHECKS")
print("--------------------------")
print(f"Saved train/validation overlap: {len(saved_overlap_train_validation):,}")
print(f"Saved additional sample vs original used rows: {len(saved_overlap_additional_original):,}")
print(f"Duplicate rows in saved training: {saved_duplicate_train:,}")
print(f"Duplicate rows in saved validation: {saved_duplicate_validation:,}")
print(f"Saved validation exactly matches original: {saved_validation_labels == original_validation_labels}")

assert len(saved_overlap_train_validation) == 0
assert len(saved_overlap_additional_original) == 0
assert saved_duplicate_train == 0
assert saved_duplicate_validation == 0
assert saved_validation_labels == original_validation_labels
assert len(saved_train) == len(combined_train_df)
assert len(saved_validation) == len(original_validation)

print("\nPASS: saved Experiment 3 train and validation files are disjoint.")
print("PASS: no additional sampled row was previously used.")
print("PASS: saved validation matches the original validation rows and labels.")


POST-SAVE INTEGRITY CHECKS
--------------------------
Saved train/validation overlap: 0
Saved additional sample vs original used rows: 0
Duplicate rows in saved training: 0
Duplicate rows in saved validation: 0
Saved validation exactly matches original: True

PASS: saved Experiment 3 train and validation files are disjoint.
PASS: no additional sampled row was previously used.
PASS: saved validation matches the original validation rows and labels.


### Finally giving a summary of the data:

In [12]:
print("=" * 62)
print("EXPERIMENT 3 DATASET PREPARATION - VERIFIED")
print("=" * 62)
print(f"Original training spectra:       {len(original_train):,}")
print(f"Additional training spectra:      {len(additional_train_df):,}")
print(f"Combined training spectra:       {len(saved_train):,}")
print(f"Original validation spectra:     {len(saved_validation):,}")
print(f"Additional sample spectra:       {len(additional_sampled_df):,}")
print(f"Additional unused spectra:       {len(additional_unused_df):,}")
print(f"Updated training classes:        {len(updated_class_to_idx):,}")
print()
print("Critical checks:")
print("  Original train/validation overlap: 0")
print("  Additional sample/original overlap: 0")
print("  Saved train/validation overlap: 0")
print("  Duplicate saved training rows: 0")
print("  Saved validation matches original: YES")
print()
print("Output files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {path.name}")
print()
print("Dataset preparation completed with all integrity checks passing.")


EXPERIMENT 3 DATASET PREPARATION - VERIFIED
Original training spectra:       91,851
Additional training spectra:      87,257
Combined training spectra:       179,108
Original validation spectra:     22,963
Additional sample spectra:       109,072
Additional unused spectra:       21,815
Updated training classes:        107,862

Critical checks:
  Original train/validation overlap: 0
  Additional sample/original overlap: 0
  Saved train/validation overlap: 0
  Duplicate saved training rows: 0
  Saved validation matches original: YES

Output files:
  additional_sampled.parquet
  additional_train.parquet
  additional_unused.parquet
  class_mapping.json
  experiment_config.json
  per_part_sampling_summary.csv
  split_manifest.csv
  train.parquet
  validation.parquet

Dataset preparation completed with all integrity checks passing.
